In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
df=pd.read_csv('/kaggle/input/comment-category-prediction-challenge/train.csv')

In [ ]:
df2=df

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print("Shape:", df.shape)

In [ ]:
print(df.isnull().sum())

In [ ]:
print((df.isnull().sum()/198000)*100)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.countplot(x="label", data=df)
plt.title("Class Distribution")
plt.show()

In [ ]:
print("Dataset is Imbalanced")

In [ ]:
df["comment"] = df["comment"].fillna("")

In [ ]:
df2['char_length'] = df['comment'].astype(str).str.len()

In [ ]:
avg_length = df2.groupby('label')['char_length'].mean()
print(avg_length)

In [ ]:
avg_length.plot(kind='bar', figsize=(8,5))

plt.title("Average Comment Length per Category")
plt.xlabel("Category")
plt.ylabel("Average Length")
plt.xticks(rotation=0)
plt.show()

In [ ]:
plt.hist(df2['char_length'], bins=80)
plt.title("Text Length Distribution")
plt.xlabel("Text Length (characters)")
plt.ylabel("Frequency")
plt.show()

In [ ]:
num_cols = [
    "emoticon_1", "emoticon_2", "emoticon_3",
    "upvote", "downvote", "if_1", "if_2"
]

In [ ]:
variance = df[num_cols].var().sort_values(ascending=False)
print("Variance:\n", variance)

plt.figure(figsize=(10,5))
sns.barplot(x=variance.index, y=variance.values)
plt.xticks(rotation=45)
plt.title("Feature Variance")
plt.show()

**Some features have very large variance, meaning their values are highly spread out compared to others. To ensure all features contribute equally to the model, scaling (StandardScaler) is applied.**

In [ ]:
plt.figure(figsize=(10,8))
corr = df[num_cols + ["label"]].corr()

sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    linewidths=0.5
)

plt.title("Correlation Heatmap")
plt.show()

**INSIGHTS**

In [ ]:
print('''Most comments are short to medium length, with a few very long outliers.
Some categories have significantly more comments than others, indicating class imbalance.
Category 3 has the lowest average comment length compared to other categories.
Category 0 dominates the dataset in terms of comment count,however,
its average comment length is shorter compared to Category 1 and Category 2,indicating that higher frequency does not necessarily imply longer content.
''')

**CLEANING COMMENTS**

In [ ]:
import re
def clean_comment(text):
    text = str(text)

    text = text.lower()                       # lowercase
    text = re.sub(r"http\S+|www\S+", "", text)  # remove urls
    text = re.sub(r"[^a-z\s]", " ", text)    # keep only letters
    text = re.sub(r"\s+", " ", text).strip() # remove extra spaces

    return text

In [ ]:
df["comment"] = df["comment"].apply(clean_comment)

In [ ]:
df['comment_length'] = df['comment'].apply(len)

In [ ]:
df = df.drop(columns=['post_id'])

In [ ]:
# df["created_date"] = pd.to_datetime(df["created_date"])

# df["month"] = df["created_date"].dt.month
# df["day_of_week"] = df["created_date"].dt.dayofweek
# df["hour"] = df["created_date"].dt.hour
# df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

# df = df.drop(columns=["created_date"])

In [ ]:
df = df.drop(columns=['created_date'])

In [ ]:
df["disability"] = df["disability"].astype(int)

In [ ]:
y = df["label"]
X = df.drop(columns=["label"])

**TRAIN TEST SPLIT**

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
num_cols = [
    "emoticon_1", "emoticon_2", "emoticon_3",
    "upvote", "downvote", "if_1", "if_2","disability","comment_length"
]

cat_cols = ["race", "religion", "gender"]
text_feature = ["comment"]



# num_cols = [
#     "emoticon_1", "emoticon_2", "emoticon_3",
#     "upvote", "downvote", "if_1", "if_2",
#     "comment_length",
#     "month", "day_of_week", "hour"
# ]

# cat_cols = ["race", "religion", "gender"]

# bool_cols = ["is_weekend"]

# text_feature = ["comment"]

**CREATING PIPELINE**

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


In [ ]:
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import FunctionTransformer

text_pipeline = Pipeline([
    ("to_1d", FunctionTransformer(lambda x: x.to_numpy().ravel(), validate=False)),
    ('tfidf', TfidfVectorizer(
        max_features=5000,
        stop_words='english',
        ngram_range=(1,2),
        min_df=5,
        max_df=0.9
    ))
])


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipeline, num_cols),
        ("cat", cat_pipeline, cat_cols),
        ("text", text_pipeline, ["comment"]),
    ],
    remainder="drop"
)

# **Baseline Model** 

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report
dummy_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DummyClassifier(strategy="most_frequent"))
])
dummy_model.fit(X_train, y_train)
y_pred = dummy_model.predict(X_val)

print("Accuracy:", accuracy_score(y_val, y_pred))
print(classification_report(y_val, y_pred, zero_division=0))

In [ ]:
from sklearn.linear_model import LogisticRegression, SGDClassifier
lgr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=2000))
])

lgr_param_grid = {
    'classifier__C': [0.01, 0.1, 1],
    'classifier__class_weight': [None, 'balanced']
}

In [ ]:
sgd_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', SGDClassifier(loss='log_loss', max_iter=1000, tol=1e-3, random_state=42))
])

sgd_param_grid = {
    'classifier__alpha': [1e-4, 1e-3, 1e-2],
    'classifier__penalty': ['l1', 'elasticnet'],
    'classifier__class_weight': [None, 'balanced']
}


# **LGR**

In [ ]:
from sklearn.model_selection import GridSearchCV
print("Tuning Logistic Regression...")
lgr_grid = GridSearchCV(
    lgr_pipeline,
    param_grid=lgr_param_grid,
    cv=3,
    n_jobs=-1,
    verbose=2,
    scoring='f1_macro'
)
lgr_grid.fit(X_train, y_train)
print("Best Logistic Regression params:", lgr_grid.best_params_)

# **SGD**

In [ ]:
print("Tuning SGD Classifier...")
from sklearn.model_selection import RandomizedSearchCV

sgd_search = GridSearchCV(
    estimator=sgd_pipeline,
    param_grid=sgd_param_grid,
    cv=3,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)
sgd_search.fit(X_train, y_train)
print("Best SGDClassifier params:", sgd_search.best_params_)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# Validation predictions
y_val_pred_lgr = lgr_grid.predict(X_val)
y_val_pred_sgd = sgd_search.predict(X_val)

# Accuracy
acc_lgr = accuracy_score(y_val, y_val_pred_lgr)
acc_sgd = accuracy_score(y_val, y_val_pred_sgd)

print("Validation Accuracy - Logistic Regression:", acc_lgr)
print("Validation Accuracy - SGDClassifier:", acc_sgd)

print("\nClassification Report - Logistic Regression")
print(classification_report(y_val, y_val_pred_lgr))

print("\nClassification Report - SGDClassifier")
print(classification_report(y_val, y_val_pred_sgd))


# **MultiNB**

In [ ]:
from sklearn.preprocessing import MinMaxScaler

num_pipeline_nb = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", MinMaxScaler())
])
preprocessor_nb = ColumnTransformer(
    transformers=[
        ("num", num_pipeline_nb, num_cols),
        ("cat", cat_pipeline, cat_cols),
        ("text", text_pipeline, ["comment"]),
    ],
    remainder="drop"
)

In [ ]:
from sklearn.naive_bayes import MultinomialNB
nb_pipeline = Pipeline([
    ("preprocessor", preprocessor_nb),
    ("classifier", MultinomialNB(alpha=0.5))
])

nb_pipeline.fit(X_train, y_train)

nb_val_score = nb_pipeline.score(X_val, y_val)

print("Naive Bayes Validation Score:", nb_val_score)

**KNN**

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=20, random_state=42)

knn_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("svd", svd),
    ("classifier", KNeighborsClassifier(n_neighbors=7)) 
])

knn_pipeline.fit(X_train, y_train)

knn_val_score = knn_pipeline.score(X_val, y_val)
print("KNN Validation Score (default params):", knn_val_score)

**SVC**

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

svm_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LinearSVC(C=1.0, max_iter=3000, random_state=42))
])


svm_pipeline.fit(X_train, y_train)

svm_val_score = svm_pipeline.score(X_val, y_val)
print("SVM Validation Score:", svm_val_score)

**COMPARING MODELS**

In [ ]:
print("Logistic-R Validation Score:", acc_lgr)
print("SGD Validation Score:", acc_sgd)
print("NB Validation Score:", nb_val_score)
print("KNN Validation Score:", knn_val_score)
print("SVM Validation Score:", svm_val_score)

In [ ]:
print("""Logistic Regression performed best (~0.89) because TF-IDF features are high-dimensional and mostly linearly separable.
SGD also performed well (~0.88) as it learns similar linear decision boundaries.
KNN achieved good accuracy (~0.83) but is sensitive to high-dimensional data.
Naive Bayes (~0.71) performed worse due to its strong independence assumption.
SVM improved after tuning (~0.79), showing it is sensitive to hyperparameters, but still underperforms compared to Logistic Regression.""")

In [ ]:
# from sklearn.ensemble import VotingClassifier

# ensemble = VotingClassifier(
#     estimators=[
#         ("lr", lgr_grid.best_estimator_),
#         ("sgd", sgd_search.best_estimator_)
#     ],
#     voting="soft"
# )

# ensemble.fit(X_train, y_train)

# ensemble_score = ensemble.score(X_val, y_val)
# print("Ensemble Score:", ensemble_score)
# Ensemble Score: 0.9028030303030303

**XGBClassifier**

In [ ]:
from xgboost import XGBClassifier

xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        n_estimators=800,
        learning_rate=0.1,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        eval_metric="mlogloss"
    ))
])

# xgb_pipeline.fit(X_train, y_train)

# xgb_val_score = xgb_pipeline.score(X_val, y_val)
# print("XGBoost Validation Score:", xgb_val_score)

**LGBMClassifier**

In [ ]:
from lightgbm import LGBMClassifier

lgbm_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LGBMClassifier(
        n_estimators=600,
        learning_rate=0.1,
        max_depth=-1,
        class_weight="balanced",
        random_state=42
    ))
])

# lgbm_pipeline.fit(X_train, y_train)

# lgbm_val_score = lgbm_pipeline.score(X_val, y_val)
# print("LightGBM Validation Score:", lgbm_val_score)

**FINAL VOTING**

In [ ]:
from sklearn.ensemble import VotingClassifier

ensemble2 = VotingClassifier(
    estimators=[
        ('lgr', lgr_grid.best_estimator_),
        ("xgbc", xgb_pipeline),
        ("lgbm", lgbm_pipeline)
    ],
    voting="soft"
)

ensemble2.fit(X_train, y_train)

ensemble2_score = ensemble2.score(X_val, y_val)
print("Ensemble2 Score:", ensemble2_score)

**MUTILAYER PERCEPTRON**

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.decomposition import TruncatedSVD

mlp_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("svd", TruncatedSVD(n_components=500, random_state=42)),
    ("classifier", MLPClassifier(
        hidden_layer_sizes=(256,128),
        activation="relu",
        solver="adam",
        alpha=0.0005,
        learning_rate_init=0.001,
        batch_size=512,
        max_iter=60,
        early_stopping=True,
        n_iter_no_change=5,
        random_state=42
    ))
])

mlp_pipeline.fit(X_train, y_train)

print("MLP Score:", mlp_pipeline.score(X_val, y_val))

In [ ]:
test_df = pd.read_csv("/kaggle/input/comment-category-prediction-challenge/test.csv")

test_df["comment"] = test_df["comment"].fillna("")
test_df["comment"] = test_df["comment"].apply(clean_comment)
test_df['comment_length'] = test_df['comment'].apply(len)
test_df = test_df.drop(columns=["post_id", "created_date"], errors='ignore')



best_model = ensemble2

test_preds = best_model.predict(test_df)

submission = pd.DataFrame({
    "ID": range(1, len(test_preds)+1),
    "label": test_preds
})

submission.to_csv("submission.csv", index=False)
print("Submission file created: submission.csv")